## Module 6 Homework
In this homework we'll put what we learned about Spark in practice.

For this homework we will be using the Yellow 2025-11 data from the official website:

## Question 1: Install Spark and PySpark
- 4.1.1

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [6]:
spark.version

'4.1.1'

In [7]:
!wget -nc https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-02 15:02:49--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:1800:b:20a5:b140:21, 2600:9000:2684:6800:b:20a5:b140:21, 2600:9000:2684:fe00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:1800:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  22.1MB/s    in 3.1s    

2026-03-02 15:02:52 (22.1 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [ ]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.show(5)  # check first 5 rows

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

## Question 2: Yellow November 2025
Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.


- 25MB



In [9]:
df_repart = df.repartition(4)

output_path = "yellow_nov2025_repart"
df_repart.write.mode("overwrite").parquet(output_path)

In [10]:
import os

folder = output_path
parquet_files = [f for f in os.listdir(folder) if f.endswith(".parquet")]

sizes = [os.path.getsize(os.path.join(folder, f)) for f in parquet_files]  # bytes
avg_size_mb = sum(sizes) / len(sizes) / (1024*1024)
print(f"Average Parquet file size: {avg_size_mb:.2f} MB")

Average Parquet file size: 25.33 MB


## Question 3: Count records
How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.


- 162,604


In [11]:
from pyspark.sql.functions import col, to_date

# Extract date only and filter
df_15 = df.filter(to_date(col("tpep_pickup_datetime")) == "2025-11-15")

# Count the records
trip_count = df_15.count()
print(f"Number of trips on 15th November: {trip_count}")

Number of trips on 15th November: 162604


## Question 4: Longest trip
What is the length of the longest trip in the dataset in hours?

- 90.6


In [14]:
from pyspark.sql.functions import col, unix_timestamp, max as spark_max

# Make sure your DataFrame is loaded
# df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

# Optional: filter out invalid trips with dropoff before pickup
df_valid = df.filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))

# Compute duration in hours
df_with_duration = df_valid.withColumn(
    "duration_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

# Get the maximum duration
max_duration = df_with_duration.agg(spark_max("duration_hours")).collect()[0][0]
print(f"Longest trip duration: {max_duration:.2f} hours")

Longest trip duration: 90.65 hours


## Question 5: User Interface
Spark's User Interface which shows the application's dashboard runs on which local port?


- 4040


In [ ]:
# Print the Spark UI Web URL
print(spark.sparkContext.uiWebUrl)

http://localhost:4040/jobs/

## Question 6: Least frequent pickup location zone
Load the zone lookup data into a temp view in Spark:

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island


In [18]:
!wget -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-02 15:18:43--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:400:b:20a5:b140:21, 2600:9000:2684:ce00:b:20a5:b140:21, 2600:9000:2684:1400:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:400:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-02 15:18:43 (33.1 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

spark = SparkSession.builder \
    .appName("YellowNov2025") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load zones CSV
zones_df = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

# Create a temporary view (optional for SQL queries)
zones_df.createOrReplaceTempView("zones")

In [20]:
# Count trips per pickup location
pickup_counts = df.groupBy("PULocationID").agg(count("*").alias("trip_count"))

# Join with zones to get zone names
pickup_with_zones = pickup_counts.join(
    zones_df,
    pickup_counts.PULocationID == zones_df.LocationID,
    how="left"
)

# Select relevant columns
pickup_with_zones = pickup_with_zones.select(
    "Zone", "Borough", "trip_count"
)

In [21]:
least_pickup = pickup_with_zones.orderBy(col("trip_count").asc()).first()
print(f"Least frequent pickup zone: {least_pickup['Zone']} with {least_pickup['trip_count']} trips")

Least frequent pickup zone: Governor's Island/Ellis Island/Liberty Island with 1 trips
